# Tugas Praktikum Sains Data Terapan (SVPL261506)
## Deteksi Visual Truk ODOL (*Overdimension Overload*) dan Kargo Berbahaya Menggunakan YOLOv11

---

### Identitas Mahasiswa
- **Nama:** Hafidz Rizqullah Prasetya
- **NIU:** 535493
- **Kelas:** PL5AA
- **Program Studi:** Sarjana Terapan Teknologi Rekayasa Perangkat Lunak (TRPL)
- **Departemen:** Teknik Elektro dan Informatika (TEDI), Sekolah Vokasi, Universitas Gadjah Mada
- **Dosen Pengampu:** Dr. Muhammad Fakhrurrifqi, S.Kom., M.Cs.

---

### Ringkasan CRISP-DM
1. **Business Understanding:** Mengatasi keterbatasan sensor jembatan timbang (WIM) yang hanya mendeteksi berat namun buta terhadap bahaya dimensi geometri muatan (truk menggunung, miring, muatan menjuntai) dengan sistem inspeksi visual berbasis AI.
2. **Data Understanding:** Dataset kustom teranotasi 1.806 citra yang mencakup kelas `roda`, `truk_normal`, dan `truk_odol`.
3. **Data Preparation:** Partisi dataset terstandarisasi (70% Train, 20% Valid, 10% Test) dengan resolusi input 416x416 dan normalisasi label format YOLO.
4. **Modeling:** Fine-tuning arsitektur YOLOv11 Nano (`yolo11n.pt`) menggunakan transfer learning.
5. **Evaluation:** Evaluasi metrik mAP@50, Precision, Recall, serta analisis Confusion Matrix.
6. **Deployment / Inferensi:** Inferensi pada unseen test dataset dan visualisasi deteksi bounding box.

## 1. Persiapan Lingkungan & Pengecekan Akselerator GPU
Memastikan Google Colab terhubung ke GPU (Nvidia T4/V100/A100) dan menginstal pustaka `ultralytics`.

In [ ]:
# Cek alokasi GPU dari Google Colab
!nvidia-smi

# Instalasi Ultralytics YOLOv11 dan dependensi pendukung
!pip install -q ultralytics pyyaml matplotlib

## 2. Pengambilan Dataset Kustom (Data Acquisition)
Mengunduh arsip dataset kustom teranotasi langsung dari repositori rilis resmi.

In [ ]:
import os
import urllib.request
import zipfile

DATASET_URL = "https://github.com/hafidzrizqullahprasetya/SDT/releases/download/v1.0-dataset/dataset_odol_ugm.zip"
ZIP_PATH = "dataset_odol_ugm.zip"
EXTRACT_DIR = "/content/dataset"

if not os.path.exists(ZIP_PATH):
    print("Mengunduh dataset kustom...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    print("Pengunduhan selesai!")

print("Mengekstrak dataset...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall("/content/")
print("Ekstraksi selesai. Direktori siap di /content/dataset")

## 3. Data Understanding & Telaah Struktur Dataset
Menganalisis distribusi partisi data (train, validation, test) dan persebaran kelas anotasi.

In [ ]:
import glob
from collections import Counter

splits = ["train", "valid", "test"]
class_map = {0: "roda", 1: "truk_normal", 2: "truk_odol"}

print("=== DISTRIBUSI FILE DATASET ===")
for s in splits:
    n_imgs = len(glob.glob(f"/content/dataset/{s}/images/*"))
    n_lbls = len(glob.glob(f"/content/dataset/{s}/labels/*"))
    print(f"Partition {s.upper():<6} -> Gambar: {n_imgs:<5} | Label: {n_lbls:<5}")

print("\n=== DISTRIBUSI KELAS ANOTASI (TRAIN SET) ===")
train_lbl_files = glob.glob("/content/dataset/train/labels/*.txt")
counts = Counter()
for f in train_lbl_files:
    with open(f, 'r') as fp:
        for line in fp:
            if line.strip():
                c = int(line.strip().split()[0])
                counts[c] += 1

for c_id, count in sorted(counts.items()):
    print(f"Kelas {c_id} ({class_map.get(c_id, 'unknown')}): {count} bounding boxes")

### Visualisasi Sampel Citra dan Bounding Box Anotasi
Menampilkan sampel citra dari data latih untuk memvalidasi ground-truth bounding box.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def draw_sample(img_path, lbl_path):
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                color = (0, 0, 255) if cls == 2 else (0, 255, 0) if cls == 1 else (255, 255, 0)
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, class_map.get(int(cls), str(cls)), (x1, max(y1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

sample_imgs = sorted(glob.glob("/content/dataset/train/images/*.jpg"))[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img_p in zip(axes, sample_imgs):
    lbl_p = img_p.replace("/images/", "/labels/").replace(".jpg", ".txt")
    ax.imshow(draw_sample(img_p, lbl_p))
    ax.axis('off')
    ax.set_title(os.path.basename(img_p))
plt.tight_layout()
plt.show()

## 4. Konfigurasi `data.yaml` untuk Lingkungan Colab
Membuat file konfigurasi pelatihan yang memetakan direktori dataset di lingkungan Colab.

In [ ]:
yaml_content = """
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 3
names:
  0: roda
  1: truk_normal
  2: truk_odol
"""

with open("/content/dataset/data_colab.yaml", "w") as f:
    f.write(yaml_content.strip())

print("Konfigurasi data_colab.yaml berhasil dibuat!")

## 5. Modeling — Pelatihan YOLOv11 (Transfer Learning)
Melatih model deteksi objek YOLOv11 Nano (`yolo11n.pt`) dengan transfer learning selama 30 epoch.

In [ ]:
from ultralytics import YOLO

# Memuat model pre-trained YOLOv11 Nano
model = YOLO("yolo11n.pt")

# Menjalankan proses training
results = model.train(
    data="/content/dataset/data_colab.yaml",
    epochs=30,
    imgsz=416,
    batch=16,
    device=0,
    name="sdt_yolo11_odol",
    project="/content/runs",
    verbose=True
)

## 6. Evaluation — Evaluasi Kinerja Model
Menilai performa model pada validation set menggunakan metrik mAP@50, Precision, dan Recall.

In [ ]:
# Validasi model menggunakan bobot terbaik (best.pt)
best_model_path = "/content/runs/sdt_yolo11_odol/weights/best.pt"
trained_model = YOLO(best_model_path)

metrics = trained_model.val(data="/content/dataset/data_colab.yaml", split="val")

print("=== METRIK EVALUASI MODEL ===")
print(f"Precision (B):  {metrics.box.mp:.4f}")
print(f"Recall (B):     {metrics.box.mr:.4f}")
print(f"mAP@50 (B):     {metrics.box.map50:.4f}")
print(f"mAP@50-95 (B):  {metrics.box.map:.4f}")

### Visualisasi Kurva Hasil Evaluasi (Confusion Matrix & F1 Curve)

In [ ]:
from IPython.display import Image, display

eval_plots = [
    "/content/runs/sdt_yolo11_odol/confusion_matrix.png",
    "/content/runs/sdt_yolo11_odol/results.png",
    "/content/runs/sdt_yolo11_odol/F1_curve.png"
]

for p in eval_plots:
    if os.path.exists(p):
        print(f"Menampilkan: {os.path.basename(p)}")
        display(Image(filename=p))
    else:
        print(f"Plot {p} belum tersedia.")

## 7. Deployment & Inferensi pada Citra Baru (Test Set)
Menguji model pada sampel citra pengujian yang belum pernah dilihat model sebelumnya (*unseen test data*).

In [ ]:
# Jalankan inferensi pada beberapa sampel data uji
test_images = sorted(glob.glob("/content/dataset/test/images/*.jpg"))[:6]
pred_results = trained_model.predict(source=test_images, conf=0.25, save=True, project="/content/runs", name="test_preds")

# Tampilkan visualisasi hasil prediksi
pred_saved_imgs = sorted(glob.glob("/content/runs/test_preds/*.jpg"))[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, p in zip(axes.flatten(), pred_saved_imgs):
    img_rgb = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.axis('off')
    ax.set_title(os.path.basename(p))
plt.tight_layout()
plt.show()

## 8. Unduh Model Terbaik (`best.pt`)
Menyimpan bobot model terbaik untuk keperluan penulisan laporan akhir praktikum atau demo lokal.

In [ ]:
from google.colab import files

print("Mengunduh file bobot model terbaik (best.pt)...")
files.download("/content/runs/sdt_yolo11_odol/weights/best.pt")